# JN7 · The re-key — finishing what you flagged

At the end of JN6b you made the grown-up call: your scorecard found the limitation you planted in JN3 (address-grouping collapsed the two buildings at 2352 Shattuck), you named it, quantified it (+69 units, one year shift), and **deferred the fix** — because re-keying the whole pipeline to a per-building identity looked like a lot of work for one building.

This notebook is where you stop deferring. You will **re-key the record to permit families**, watch 2352 Shattuck resolve itself — and then discover something better than a fix: the deferral's justification was **wrong**. It was never one building. And the tool that proves it is one you've been holding since the very first notebook.

**Curriculum notebook 7 of 7 — the final act.** Clonable + read-only on the source data; it writes one output file (your adjudication ledger).

### Running the cells

To run a cell, click it and press **Shift + Return**, or click the **run (▸) button** on the cell. The simplest way through any notebook here is to start at the top and run each cell in order, reading the output that appears beneath it.

Some of the computational cells may look complex right now — that's expected, and it's fine. **You don't need to understand every line yet;** the ideas become clear as you go. Run them, watch what they produce, and keep moving.

💡 Tip: the **Next** link opens the following notebook in a new tab. If Colab says you have too many sessions, just close the previous tab and continue.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN6b · Join, decompose, score](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN6b_join_score.ipynb)  |  [🎉 You've finished the series — back to the course](https://berkeleybuild.com/data-science-curriculum.html)

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally** (it detects a checkout and skips). On Colab / a bare session it recreates the minimal repo layout under the working directory so the config cell below finds everything unchanged.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')

local repo detected - no fetch needed


In [2]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## Config

In [3]:
# === CONFIG (clonable) ===
from pathlib import Path
import sys, glob
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PERMIT_GLOB = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
ORACLE_DB   = str(REPO_ROOT / 'databases/hcd_apr_mirror_2026-06-17_fresh.db')
OUTDIR      = REPO_ROOT / 'notebooks/curriculum/output'; OUTDIR.mkdir(parents=True, exist_ok=True)
HEADER_ROW  = 7
sys.path.insert(0, str(REPO_ROOT / 'scripts')); sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root:', REPO_ROOT)

repo root: /Users/johngage/berkeley-data


# PHASE 7a — The re-key

**The idea, in one line: an address says *where*; a permit family says *what got built*.**

You have been touching permit families the whole course. In **JN00** you found 2150 Kittredge wearing a dozen permits, each repeating its 169 homes. In **JN4** you imported `extract_master_permit` — the function that reads `B2019-05574-DEF03` and answers "that belongs to `B2019-05574`." A **family** is a base permit plus all its revisions and deferrals: one construction project's whole paper trail.

So far the spine has grouped permits by **address** and kept the MAX units. Now we flip it: group by **family**. The address becomes *evidence attached to* a building instead of the building's *identity*.

**Our plan:** rebuild completions exactly as before — same loading, same `is_housing`, same `net_units` — changing only the key: `extract_master_permit(PermitNumber)` instead of the address.

In [4]:
import pandas as pd
from collections import defaultdict
from housing_predicates import is_housing, net_units
from s0_keys import normalize_address
from cpra_dedup import extract_master_permit
def pdate(x):
    d = pd.to_datetime(str(x), errors='coerce'); return d.date() if pd.notna(d) else None
def load(p):
    d = pd.read_excel(p, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
df = pd.concat([load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)
df = df[df['PermitNumber'].notna()].rename(columns={'Finaled Date': 'FinaledDate'}).copy()
df['isnew'] = df['Work Type'].astype(str).str.strip() == 'New'
df = df[[is_housing(o,u,n,a) for o,u,n,a in zip(df['OccType'],df['UnitsAdded'],df['NumberUnits'],df['ADU'])]]

# THE RE-KEY: one entry per permit FAMILY (the master permit), not per address
fam = defaultdict(lambda: {'units': 0.0, 'final': [], 'addr': set(), 'desc': '', 'adu': False})
for r in df.itertuples(index=False):
    m = extract_master_permit(str(r.PermitNumber))
    f = fam[m]
    f['units'] = max(f['units'], net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU))
    st = r.StreetType; st = '' if (st is None or str(st).strip().lower() == 'nan') else str(st)
    k = normalize_address(f'{r.StreetNumber} {r.StreetName} {st}'.strip())
    if k.number: f['addr'].add((k.number, k.street, k.stype))
    if str(r.PermitNumber) == m:                       # facts about the family come from its master row
        d_ = pdate(r.FinaledDate)
        if d_: f['final'].append(d_)
        if not f['desc']: f['desc'] = str(r.WorkDescription)
        f['adu'] = f['adu'] or str(r.ADU).strip().lower().startswith('y')

families = {m: f for m, f in fam.items() if f['units'] > 0 and f['final']}   # completed, home-adding
print(f'completed permit families : {len(families):,}   face-value units: {int(sum(f["units"] for f in families.values())):,}')
print( 'the address-keyed spine   : 951     face-value units: 4,310   (JN6b, same data)')

completed permit families : 1,032   face-value units: 4,805
the address-keyed spine   : 951     face-value units: 4,310   (JN6b, same data)


In [5]:
md(f'''## What just happened

Same data, same predicates, one changed key — and the record got **bigger**: **{len(families):,}** completed families carrying **{int(sum(f["units"] for f in families.values())):,}** face-value units, versus the address spine's **951** buildings and **4,310** units. How can changing the *key* create units?

It didn't create them — it **revealed** them. The address key silently took the MAX across everything at an address: multiple genuinely different projects at one address collapsed into one number. The family key gives every construction project its own row, which surfaces real buildings the MAX was swallowing — **and** surfaces every duplicate and junk record the MAX was *also* conveniently hiding. Visibility cuts both ways; that is the honest price of a better identity, and the rest of this notebook is about paying it.''')

## What just happened

Same data, same predicates, one changed key — and the record got **bigger**: **1,032** completed families carrying **4,805** face-value units, versus the address spine's **951** buildings and **4,310** units. How can changing the *key* create units?

It didn't create them — it **revealed** them. The address key silently took the MAX across everything at an address: multiple genuinely different projects at one address collapsed into one number. The family key gives every construction project its own row, which surfaces real buildings the MAX was swallowing — **and** surfaces every duplicate and junk record the MAX was *also* conveniently hiding. Visibility cuts both ways; that is the honest price of a better identity, and the rest of this notebook is about paying it.

# PHASE 7b — 2352 Shattuck, resolved (and a surprise)

The building you flagged. Under the address key it was ONE 135-unit building dated 2023. The city said TWO buildings: 135 @ 2022 and 69 @ 2023. Let's look at what the **family** key says.

In [6]:
k = normalize_address('2352 Shattuck Ave')
sh = {m: f for m, f in families.items() if (k.number, k.street, k.stype) in f['addr']}
rows = [{'family': m, 'units': int(f['units']), 'finaled': max(f['final']).isoformat(),
         'description': f['desc'][:78]} for m, f in sorted(sh.items())]
pd.DataFrame(rows)

,family,units,finaled,description
0,B2019-05574,135,2022-01-14,Phase II of II - North Building; Structural Su...
1,B2019-05575,69,2023-08-08,Phase I - South Building: An eight-story mixe...
2,B2021-03302,69,2023-08-08,"Phase II of South Building: Architectural, Str..."


In [7]:
md(f'''## What just happened

The re-key found **{len(sh)}** completed families at 2352 Shattuck — where the address key saw one building and the city reported two.

Two of them are the city's two buildings, instantly: **B2019-05574 = 135 units finaled 2022** (the North building — the year misattribution is *gone*; the family's own master permit carries its own date) and a **69-unit South building finaled 2023** (the +69 miss is *gone*). The flag you filed in JN6b — both halves — dissolved by changing the key. That is what "fix it at the identity layer" means.

But look again: there are **two** 69-unit families, finaled the **same day**. Did we just trade a missed building for an invented one? Read the descriptions — the data answers the question itself: *"**Phase I** - South Building"* (B2019-05575) and *"**Phase II** of South Building"* (B2021-03302). One real building, permitted in two phases, each phase a full permit family carrying the full 69 units. The re-key is a better identity — and it is still **not the truth by itself.**''')

## What just happened

The re-key found **3** completed families at 2352 Shattuck — where the address key saw one building and the city reported two.

Two of them are the city's two buildings, instantly: **B2019-05574 = 135 units finaled 2022** (the North building — the year misattribution is *gone*; the family's own master permit carries its own date) and a **69-unit South building finaled 2023** (the +69 miss is *gone*). The flag you filed in JN6b — both halves — dissolved by changing the key. That is what "fix it at the identity layer" means.

But look again: there are **two** 69-unit families, finaled the **same day**. Did we just trade a missed building for an invented one? Read the descriptions — the data answers the question itself: *"**Phase I** - South Building"* (B2019-05575) and *"**Phase II** of South Building"* (B2021-03302). One real building, permitted in two phases, each phase a full permit family carrying the full 69 units. The re-key is a better identity — and it is still **not the truth by itself.**

### Identity is adjudication, not a groupby

Here is the lesson this whole notebook exists to teach. No key — address, family, parcel, anything — mechanically equals "one building." The **evidence** decides: these two families share an address, carry identical unit counts, finaled the **same day**, and their own descriptions say Phase I / Phase II of the *same named building*. That is a **merge**: one building, counted once.

We record the call — with its evidence — and move on. (Which family "survives" is a bookkeeping choice; what matters is that the decision and its provenance are written down. You'll see where they get written in Phase 7d.)

In [8]:
MERGED_PHASES = {'B2021-03302': 'B2019-05575'}   # Phase II folds into Phase I - South Building
rekeyed = {m: f for m, f in families.items() if m not in MERGED_PHASES}
sh2 = {m: f for m, f in rekeyed.items() if (k.number, k.street, k.stype) in f['addr']}
mine_shattuck = sorted((max(f['final']).year, int(f['units'])) for f in sh2.values())
print('my re-keyed 2352 Shattuck :', mine_shattuck)
print("the city's record         : [(2022, 135), (2023, 69)]")
assert mine_shattuck == [(2022, 135), (2023, 69)]
print('CHECKPOINT 7a PASS - the planted limitation is RESOLVED: both buildings, both years, exact match')

my re-keyed 2352 Shattuck : [(2022, 135), (2023, 69)]
the city's record         : [(2022, 135), (2023, 69)]
CHECKPOINT 7a PASS - the planted limitation is RESOLVED: both buildings, both years, exact match


# PHASE 7c — Was it ever "one building"?

JN6b's deferral rested on one claim: the re-key is real work *"for one building — the only such case in the data."* Now you can test that claim in one line: how many addresses hold **two or more** completed families?

In [9]:
by_addr = defaultdict(list)
for m, f in rekeyed.items():
    for a in f['addr']: by_addr[a].append(m)
multi = {a: ms for a, ms in by_addr.items() if len(ms) > 1}
print(f'addresses holding 2+ completed families: {len(multi)}   (the deferral said: one)')

SPECIMENS = ['1173 HEARST AVE', '1030 GRAYSON ST', '1136 KEITH AVE', '1260 HOPKINS ST', '1219 DERBY ST']
rows = []
for addr in SPECIMENS:
    ak = normalize_address(addr)
    for m_ in sorted(by_addr.get((ak.number, ak.street, ak.stype), [])):
        f = rekeyed[m_]
        rows.append({'address': addr, 'family': m_, 'units': int(f['units']),
                     'finaled': max(f['final']).isoformat(), 'ADU': f['adu'],
                     'description': f['desc'][:64]})
pd.DataFrame(rows)

addresses holding 2+ completed families: 62   (the deferral said: one)


,address,family,units,finaled,ADU,description
0,1173 HEARST AVE,B2020-02941,2,2023-03-06,False,Construction of new duplex. There is an exist...
1,1173 HEARST AVE,B2020-02942,2,2023-03-15,False,1721 SF - Construction of a new duplex. There ...
2,1030 GRAYSON ST,B2017-00518,2,2018-10-17,False,New Front (Northerly) Duplex (Address assignme...
3,1030 GRAYSON ST,B2017-03295,2,2018-10-17,False,New Rear (southern) Duplex (Address assignment...
4,1136 KEITH AVE,B2024-02570,1,2025-11-10,False,Building new Single Family Residence. (See dem...
5,1136 KEITH AVE,B2024-02712,1,2025-08-20,False,Temporary Foundation for a 2950 sq. ft. single...
6,1260 HOPKINS ST,B2023-04482,2,2024-02-01,True,Installation of water heater in Apartment 8.
7,1260 HOPKINS ST,B2024-00662,2,2025-06-04,True,Installation of water heater in #11
8,1260 HOPKINS ST,B2025-01553,2,2025-06-04,True,Installation of seismic safety shut off valve
9,1219 DERBY ST,B2024-01564,1,2025-04-24,True,Detached 377 sq ft ADU


In [10]:
md(f'''## What just happened

**{len(multi)} addresses** — not one. The deferral's justification was factually wrong, and your own re-keyed record just proved it. Read the specimens: they are not one problem, they are **three different problems demanding three OPPOSITE calls**:

1. **Real pairs — KEEP BOTH.** 1173 Hearst is two genuinely different duplexes (two families, two dates). 1030 Grayson is a front duplex and a rear duplex — four real homes. Merging these would *erase real housing* (the live project caught real backyard ADUs minutes from being merged away exactly this way).
2. **Phases — MERGE.** 1136 Keith has a "temporary foundation" permit *and* the house it held up — one home, two families, just like Shattuck's South building.
3. **Maintenance wearing unit counts — DROP.** 1260 Hopkins' families are *water-heater installations* in an apartment building; 1219 Derby's second family is a *dual-meter electrical service*. Each "carries" the property's 2 units — but a water heater completes no homes. (Remember this class: you will meet it again on the city's side of the table in a moment.)

Same signal — "multiple families, one address" — three opposite adjudications, decided by **evidence** (descriptions, dates, the ADU flag), not by any key. *This* is why the fix was real work, and why the informed flag in JN6b was still the right call at the time: you can't re-key responsibly until you're ready to adjudicate what the re-key reveals.''')

## What just happened

**62 addresses** — not one. The deferral's justification was factually wrong, and your own re-keyed record just proved it. Read the specimens: they are not one problem, they are **three different problems demanding three OPPOSITE calls**:

1. **Real pairs — KEEP BOTH.** 1173 Hearst is two genuinely different duplexes (two families, two dates). 1030 Grayson is a front duplex and a rear duplex — four real homes. Merging these would *erase real housing* (the live project caught real backyard ADUs minutes from being merged away exactly this way).
2. **Phases — MERGE.** 1136 Keith has a "temporary foundation" permit *and* the house it held up — one home, two families, just like Shattuck's South building.
3. **Maintenance wearing unit counts — DROP.** 1260 Hopkins' families are *water-heater installations* in an apartment building; 1219 Derby's second family is a *dual-meter electrical service*. Each "carries" the property's 2 units — but a water heater completes no homes. (Remember this class: you will meet it again on the city's side of the table in a moment.)

Same signal — "multiple families, one address" — three opposite adjudications, decided by **evidence** (descriptions, dates, the ADU flag), not by any key. *This* is why the fix was real work, and why the informed flag in JN6b was still the right call at the time: you can't re-key responsibly until you're ready to adjudicate what the re-key reveals.

⚠ *Honesty note on the number:* the multi-family address count above is a **face-value** count — some entries are themselves artifacts of the `UnitsAdded`/`NumberUnits` quirks you met in JN3 (that's *why* several specimens are maintenance permits wearing unit counts). The exact count moves as the counting improves; what's load-bearing is that the phenomenon is a **class**, not a one-off — and that each member needs an evidence-based call, not a formula.

# PHASE 7d — The ledger: never adjudicate the same thing twice

You've now made four judgment calls (one merge at Shattuck, and the three specimen classes). Where do decisions like that *live*? If the answer is "in your memory" or "in a chat log," you will re-derive them — badly — six months from now. The live project learned this the hard way, then built the fix: an **adjudication ledger**. Every resolved question becomes a row: what was decided, the number it grounds, and the **evidence** — so a decision, once made, is never silently re-made.

**Our plan:** write your four calls to a small CSV, each with its provenance. This is a teaching miniature of the real thing (the project's ledger holds ~190 rows grounding ~597 units, each row citing the document that decided it).

In [11]:
ledger = pd.DataFrame([
    {'address': '2352 Shattuck Ave', 'families': 'B2019-05575 + B2021-03302', 'decision': 'merge_phases',
     'grounded_homes': 69, 'evidence': 'same address, same 69u, finaled same day; descriptions: "Phase I - South Building" / "Phase II of South Building"'},
    {'address': '1173 Hearst Ave',   'families': 'B2020-02941 + B2020-02942', 'decision': 'keep_both',
     'grounded_homes': 4,  'evidence': 'two distinct duplex constructions, separate finaled dates; merging would erase real homes'},
    {'address': '1136 Keith Ave',    'families': 'B2024-02570 + B2024-02712', 'decision': 'merge_phases',
     'grounded_homes': 1,  'evidence': 'B2024-02712 is "Temporary Foundation for a ... single family Residence" - a phase of B2024-02570, not a second home'},
    {'address': '1260 Hopkins St',   'families': 'B2023-04482 + B2024-00662', 'decision': 'drop_units',
     'grounded_homes': 0,  'evidence': 'water-heater installations wearing the property unit count; maintenance completes no homes'},
], columns=['address', 'families', 'decision', 'grounded_homes', 'evidence'])
ledger['as_of'] = '2026-07-03'   # the date each call was made - provenance, part of the record
out = OUTDIR / 'my_adjudications.csv'
ledger.to_csv(out, index=False)
print('wrote', out)
ledger[['address', 'decision', 'grounded_homes', 'evidence']]

wrote /Users/johngage/berkeley-data/notebooks/curriculum/output/my_adjudications.csv


,address,decision,grounded_homes,evidence
0,2352 Shattuck Ave,merge_phases,69,"same address, same 69u, finaled same day; desc..."
1,1173 Hearst Ave,keep_both,4,"two distinct duplex constructions, separate fi..."
2,1136 Keith Ave,merge_phases,1,"B2024-02712 is ""Temporary Foundation for a ......"
3,1260 Hopkins St,drop_units,0,water-heater installations wearing the propert...


Three disciplines make a ledger worth more than a note-to-self, and they're visible in those four rows:

- **Every row carries its evidence.** A decision without provenance is just an opinion with a timestamp — and a future re-derivation waiting to happen. With the quote from the permit record *in the row*, anyone (including you, next year) can re-check the call without redoing the work.
- **Append-only.** When a call turns out wrong (the live project has retracted three), you don't erase the row — you add the correction with *its* evidence. The ledger is a history of what was believed and why, not just a current-state table.
- **The ledger is memory.** Before investigating any discrepancy, check the ledger first: *has this already been adjudicated?* The single most expensive failure mode in this work is re-deriving, from scratch, an answer the project already earned.

# PHASE 7e — The city errs in classes too

One more payoff before the ending. In JN6a you caught the city double-submitting CY2025. In 7c you met *our* side's junk class (maintenance permits wearing unit counts). Here's the closing symmetry: **the city's filing has mechanical error classes of its own** — and two of them are findable, right now, in the oracle you already have, with a few lines each.

**Class 1 — cross-year re-credit:** the same building permit's units claimed as completed in *two different years*. **Class 2 — approval credited as completion:** a *Planning* record (a `ZP…` zoning approval — permission to build) filed as a certificate of occupancy (proof it *was* built). Entitled ≠ built.

In [12]:
import sqlite3
con = sqlite3.connect(f'file:{ORACLE_DB}?mode=ro', uri=True)
cols = [r[1] for r in con.execute('PRAGMA table_info(table_a2)')]
CO_SUM = '+'.join(f"CAST(NULLIF({c},'') AS INT)" for c in cols if c.startswith('CO_') and 'DT' not in c)
oc = pd.read_sql(f"SELECT YEAR, JURS_TRACKING_ID id, STREET_ADDRESS addr, ({CO_SUM}) co "
                 f"FROM table_a2 WHERE CO_ISSUE_DT1 <> ''", con)
oc = oc[oc.co.fillna(0) > 0]

# class 1: same B-permit with completed units in 2+ distinct years
b = oc[oc.id.astype(str).str.match(r'^B\d{4}-\d{4,5}')]
g = b.groupby('id').YEAR.nunique()
cross = b[b.id.isin(g[g > 1].index)].sort_values(['id', 'YEAR'])
print('cross-year re-credits:')
print(cross[['id', 'YEAR', 'addr', 'co']].to_string(index=False))

# class 2: Planning-approval ids (ZP/UP/DR...) carrying completion credit
zp = oc[oc.id.astype(str).str.match(r'^(ZP|UP|DR|AP|LM)\d{4}-\d+')]
print('\napprovals credited as completions:')
print(zp[['id', 'YEAR', 'addr', 'co']].to_string(index=False))

assert 'B2022-02049' in set(cross.id), 'anchor miss: the documented cross-year instance'
assert 'ZP2019-0022' in set(zp.id),    'anchor miss: the documented approval-as-CO instance'
print('\nCHECKPOINT 7b PASS - both documented anchor instances found by YOUR detectors')

cross-year re-credits:
         id YEAR              addr  co
B2018-02288 2019   1711 Mlk Jr Way   2
B2018-02288 2020  1711 M L King Jr   1
B2019-03765 2020  1811 Sixty-Third   2
B2019-03765 2021  1811 Sixty-Third   2
B2022-02049 2023 1825 Berkeley (1)   1
B2022-02049 2024 1825 BERKELEY Way   1

approvals credited as completions:
         id YEAR        addr  co
ZP2018-0086 2021  1526 Sixth   1
ZP2019-0022 2019 1284 Hearst   1

CHECKPOINT 7b PASS - both documented anchor instances found by YOUR detectors


In [13]:
md(f'''## What just happened

Your two little filters flagged **{cross.id.nunique()}** cross-year re-credits and **{zp.id.nunique()}** approvals-filed-as-completions — including the two *documented, receipted* instances the live project adjudicated (`B2022-02049`, `ZP2019-0022`) **and** the very suspects its automated watcher queued for review the week this notebook was written (`B2018-02288`, `ZP2018-0086`). You are running the real audit's real detectors on the real filing.

The deeper point: cities don't err at random — they err in **mechanical classes**, which means the checks can be mechanical too, run by anyone, forever. The project documented five such classes; you just ran two. The other three are yours as exercises: **full-row duplicates** (you already met the CY2025 double-submission in JN6a), **completion dated at issuance** (`B2019-03765` sits in your cross-year table above — pull its permit dates from the feed and see why), and **the meter re-credit** — the city crediting a *utility-meter permit* with the homes it serves. You met that exact class on OUR side at 1219 Derby. Both ledgers, same trap.

**Discipline note:** a flag is a *suspect*, not a verdict — every one of these went through adjudication (evidence, ledger row, receipt) before the project counted it as a city error. Detectors queue questions; they never answer them.''')

## What just happened

Your two little filters flagged **3** cross-year re-credits and **2** approvals-filed-as-completions — including the two *documented, receipted* instances the live project adjudicated (`B2022-02049`, `ZP2019-0022`) **and** the very suspects its automated watcher queued for review the week this notebook was written (`B2018-02288`, `ZP2018-0086`). You are running the real audit's real detectors on the real filing.

The deeper point: cities don't err at random — they err in **mechanical classes**, which means the checks can be mechanical too, run by anyone, forever. The project documented five such classes; you just ran two. The other three are yours as exercises: **full-row duplicates** (you already met the CY2025 double-submission in JN6a), **completion dated at issuance** (`B2019-03765` sits in your cross-year table above — pull its permit dates from the feed and see why), and **the meter re-credit** — the city crediting a *utility-meter permit* with the homes it serves. You met that exact class on OUR side at 1219 Derby. Both ledgers, same trap.

**Discipline note:** a flag is a *suspect*, not a verdict — every one of these went through adjudication (evidence, ledger row, receipt) before the project counted it as a city error. Detectors queue questions; they never answer them.

# The ending — where you actually are

In JN6b the honest scorecard said: *we differ by a decomposable amount, every unit named into buckets.* The live project, using exactly the method you now own — the re-key you just performed, the adjudication ledger you just wrote, the detectors you just ran, times about a hundred and ninety receipted decisions — pushed that to its terminal state: **4,229 permit-derived completed homes (2018–2025) versus the city's adjudicated 4,099 — a +130 difference in which every row, in both directions, is named** (as of July 2026).

That result is public: **[berkeleybuild.com/housing-audit.html](https://berkeleybuild.com/housing-audit.html)** — the Audit page. Read it now. Every line on it was produced by things you have personally done across these notebooks: ingest, key, count, date, join, decompose, re-key, adjudicate, detect. And it doesn't end — the filing *moves*, so the project's watcher re-pulls it, diffs it, and runs your Phase-7e detectors on schedule. An audit is a practice, not a document.

**One last thing.** Back in JN00 you asked an AI to read this dataset, and you saved its answer. Go get it. Grade it — what did it see, what did it miss, what did it *assert* that you now know needs a ledger row? That answer was your first oracle. You have spent seven notebooks learning what to do with oracles: **interrogate them, name every difference, write down the evidence — and never let an answer, however confident, skip adjudication.**

That is the whole method. It now runs in your hands, on any city that publishes its permits.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN6b · Join, decompose, score](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN6b_join_score.ipynb)  |  [🎉 You've finished the series — back to the course](https://berkeleybuild.com/data-science-curriculum.html)